In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

#India Map Variables and Colors
variables = ["AP", "AR", "AS", "BR", "CG", "GA", "GJ", "HR", "HP", "JH", "KA", "KL", "MP", "MH", "MN", "ML", "MZ", "NL", "OD", "PB", "RJ", "SK", "TN", "TS", "TR", "UP", "UK", "WB", "JK"]
colors = ['Red', 'Green', 'Blue', 'Yellow']

#India Map Adjacency List
neighbors = {
    "AP": ["TS", "OD", "TN", "KA"],
    "AR": ["AS", "NL"],
    "AS": ["AR", "NL", "ML", "TR", "MZ", "MN", "WB"],
    "BR": ["UP", "JH", "WB"],
    "CG": ["UP", "JH", "OD", "MH"],
    "GA": ["MH", "KA"],
    "GJ": ["MH", "RJ"],
    "HR": ["PB", "HP", "UK", "RJ"],
    "HP": ["JK", "PB", "HR", "UK"],
    "JH": ["BR", "UP", "CG", "OD", "WB"],
    "KA": ["MH", "AP", "TS", "TN", "KL", "GA"],
    "KL": ["KA", "TN"],
    "MP": ["RJ", "UP", "CG", "MH", "GJ"],
    "MH": ["GJ", "MP", "CG", "TS", "KA", "GA"],
    "MN": ["AS", "MZ", "NL"],
    "ML": ["AS", "TR"],
    "MZ": ["AS", "MN", "TR"],
    "NL": ["AR", "AS", "MN"],
    "OD": ["WB", "JH", "CG", "AP", "TS"],
    "PB": ["JK", "HP", "HR", "RJ"],
    "RJ": ["PB", "HR", "MP", "GJ", "UP"],
    "SK": ["WB"],
    "TN": ["AP", "KA", "KL"],
    "TS": ["MH", "KA", "AP", "OD"],
    "TR": ["AS", "ML", "MZ"],
    "UP": ["UK", "HR", "RJ", "MP", "CG", "JH", "BR"],
    "UK": ["HP", "HR", "UP"],
    "WB": ["BR", "JH", "OD", "AS", "SK"],
    "JK": ["HP", "PB"] 
}

#Utility Functions (is_valid)
def is_valid(assignment, var, value):
    for n in neighbors[var]:
        if n in assignment and assignment[n] == value:
            return False
    return True

#Plain Backtracking
plain_steps = 0
def backtrack_plain(assignment):
    global plain_steps
    plain_steps += 1
    if len(assignment) == len(variables):
        return assignment
    
    for var in variables:
        if var not in assignment:
            break
            
    for value in colors:
        if is_valid(assignment, var, value):
            assignment[var] = value
            result = backtrack_plain(assignment)
            if result:
                return result
            del assignment[var]
            
    return None

#Heuristics (MRV and LCV)
def select_unassigned_var(assignment):
    unassigned = [v for v in variables if v not in assignment]
    return min(unassigned, key=lambda var: sum(is_valid(assignment, var, c) for c in colors))

def order_domain_values(var, assignment):
    def conflicts(val):
        return sum(1 for n in neighbors[var] if n not in assignment and not is_valid({**assignment, var: val}, n, val))
    return sorted(colors, key=conflicts)

#Backtracking with MRV + LCV
heuristic_steps = 0
def backtrack_heuristic(assignment):
    global heuristic_steps
    heuristic_steps += 1
    if len(assignment) == len(variables):
        return assignment
    
    var = select_unassigned_var(assignment)
    for value in order_domain_values(var, assignment):
        if is_valid(assignment, var, value):
            assignment[var] = value
            result = backtrack_heuristic(assignment)
            if result:
                return result
            del assignment[var]
            
    return None

#AC-3 and Backtracking with Heuristics + AC-3
def revise(queue, domains, var1, var2):
    revised = False
    new_domain_v1 = []
    for val1 in domains[var1]:
        has_support = False
        for val2 in domains[var2]:
            if val1 != val2:
                has_support = True
                break
        if has_support:
            new_domain_v1.append(val1)
        else:
            revised = True
            
    domains[var1] = new_domain_v1
    return revised

def ac3(domains):
    queue = []
    for var1 in variables:
        for var2 in neighbors[var1]:
            queue.append((var1, var2))

    while queue:
        var1, var2 = queue.pop(0)
        if revise(queue, domains, var1, var2):
            if not domains[var1]:
                return False
            for var3 in neighbors[var1]:
                if var3 != var2:
                    queue.append((var3, var1))
    return True

heuristic_ac3_steps = 0
def backtrack_ac3(assignment, domains):
    global heuristic_ac3_steps
    heuristic_ac3_steps += 1
    
    if len(assignment) == len(variables):
        return assignment
    
    var = select_unassigned_var(assignment)
    
    for value in order_domain_values(var, assignment):
        if is_valid(assignment, var, value):
            assignment[var] = value
            
            new_domains = {v: d[:] for v, d in domains.items()}
            new_domains[var] = [value]
            
            if ac3(new_domains):
                result = backtrack_ac3(assignment, new_domains)
                if result:
                    return result
            
            del assignment[var]
            
    return None

#Visualization
def visualize_coloring(solution, title):
    G = nx.Graph()
    
    for var in variables:
        G.add_node(var)
    for var, neighs in neighbors.items():
        for n in neighs:
            G.add_edge(var, n)
            
    pos = {
        "JK": (4, 11), "HP": (4.5, 9.5), "PB": (3.5, 9), "UK": (5, 9), "HR": (4, 8),
        "RJ": (3, 7.5), "UP": (5, 7.5), "BR": (6.5, 7), "JH": (6, 6), "WB": (7.5, 7),
        "GJ": (2, 6.5), "MP": (4, 6), "CG": (5, 5), "OD": (6.5, 5), "SK": (8, 8.5),
        "MH": (3, 4.5), "TS": (4.5, 4), "KA": (3, 3), "AP": (5, 3), "GA": (2.5, 3),
        "KL": (3, 1), "TN": (4, 1), "AS": (8.5, 8.5), "AR": (10, 8.5), "NL": (9.5, 7.5),
        "MN": (9, 6.5), "MZ": (9.5, 5.5), "TR": (8, 6), "ML": (8, 7)
    }
    
    node_colors = [solution.get(node, "white") for node in G.nodes()]
    
    plt.figure(figsize=(10, 10))
    nx.draw(
        G, pos,
        with_labels=True,
        node_color=node_colors,
        node_size=1500,
        font_size=10,
        font_weight="bold",
        edge_color="gray"
    )
    plt.title(title)
    plt.show()

#Main Block
if __name__ == "__main__":
    # Plain Backtracking
    plain_solution = backtrack_plain({})
    print("Plain Backtracking Solution:", plain_solution)
    print("Plain Backtracking Steps:", plain_steps)
    visualize_coloring(plain_solution, f"Plain Backtracking ({plain_steps} steps)")

    print("\n" + "="*50 + "\n")

    # Backtracking with MRV + LCV
    heuristic_solution = backtrack_heuristic({})
    print("Backtracking with MRV + LCV Solution:", heuristic_solution)
    print("Backtracking with MRV + LCV Steps:", heuristic_steps)
    visualize_coloring(heuristic_solution, f"MRV + LCV Heuristic ({heuristic_steps} steps)")

    print("\n" + "="*50 + "\n")

    # Backtracking with MRV + LCV + AC-3
    initial_domains = {var: list(colors) for var in variables}
    ac3_solution = backtrack_ac3({}, initial_domains)
    print("Backtracking with MRV + LCV + AC-3 Solution:", ac3_solution)
    print("Backtracking with MRV + LCV + AC-3 Steps:", heuristic_ac3_steps)
    visualize_coloring(ac3_solution, f"MRV + LCV + AC-3 ({heuristic_ac3_steps} steps)")

: 